# 09 — Análisis NLP: BERTopic + Clasificación Zero-Shot

Ejecuta el pipeline completo de NLP sobre `scam_us_FINAL_v8.csv`.

**Fase 1 — BERTopic:** tópicos emergentes no supervisados (~5-10 min).
**Fase 2 — BART-MNLI zero-shot:** clasificación en 14 categorías (~30-90 min).

## Antes de Run All

- Mac enchufado.
- `caffeinate -dimsu` corriendo en otra terminal.
- Kernel del notebook: **Python (tfm-nlp)**.


## Verificación de que el entorno está bien

In [ ]:
# Verificación rápida de que todo está OK antes de empezar
import sys
print(f"Python: {sys.version}")

import numpy, torch, transformers, sentence_transformers, bertopic, pandas
print(f"numpy:                 {numpy.__version__}")
print(f"torch:                 {torch.__version__}")
print(f"transformers:          {transformers.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"bertopic:              {bertopic.__version__}")
print(f"pandas:                {pandas.__version__}")
print()
print("✓ Entorno listo")


## Carga del corpus

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import time

pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_us_FINAL_v8.csv")
print(f"Tweets cargados: {len(df)}")

# Preparar texto limpio
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")

def clean_for_analysis(text):
    if not isinstance(text, str):
        return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean_for_analysis)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    mask_empty = df["clean_text"] == ""
    df.loc[mask_empty, "clean_text"] = df.loc[mask_empty, "text"].apply(clean_for_analysis)

# Filtrar tweets muy cortos
df = df[df["clean_text"].str.len() >= 20].reset_index(drop=True)
print(f"Tras filtrar textos cortos: {len(df)}")

docs = df["clean_text"].tolist()


## FASE 1 — BERTopic

⏳ ~5-10 minutos. Descarga inicial del modelo all-MiniLM-L6-v2 (~80 MB).


In [ ]:
print("⏳ Iniciando BERTopic...")
t0 = time.time()

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Para BERTopic 0.16+ con sentence-transformers 2.7, mejor pasar el nombre del modelo
# y dejar que BERTopic gestione los submodelos por defecto.
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    min_topic_size=30,           # adaptado a corpus de ~2k
    nr_topics="auto",            # reduce automáticamente tópicos similares
    calculate_probabilities=False,
    language="english",
    verbose=True,
)

print("⏳ Calculando embeddings y agrupando...")
topics, _ = topic_model.fit_transform(docs)

t_bertopic = time.time() - t0
n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = sum(1 for t in topics if t == -1)
print(f"\n✓ BERTopic completado en {t_bertopic/60:.1f} min")
print(f"  Tópicos encontrados: {n_topics}")
print(f"  Tweets en outliers (-1): {n_outliers}")


In [ ]:
# Información detallada de los tópicos
topic_info = topic_model.get_topic_info()
print("=== INFO DE TÓPICOS ===")
print(topic_info.head(25))


In [ ]:
# Asignar tópico y keywords a cada tweet
df["bertopic_id"] = topics

# Diccionario topic_id → keywords
topic_keywords = {}
for tid in set(topics):
    if tid == -1:
        topic_keywords[tid] = "outliers"
        continue
    words = topic_model.get_topic(tid)
    if words:
        topic_keywords[tid] = ", ".join([w for w, _ in words[:8]])
    else:
        topic_keywords[tid] = ""

df["bertopic_keywords"] = df["bertopic_id"].map(topic_keywords)

print("=== RESUMEN BERTOPIC ===\n")
for tid in sorted(set(topics)):
    n = (df["bertopic_id"] == tid).sum()
    kw = topic_keywords.get(tid, "")
    print(f"  Tópico {tid:>3} ({n:>4} tweets):  {kw[:80]}")


In [ ]:
# Ejemplos por tópico
print("=== EJEMPLOS POR TÓPICO ===\n")
shown = 0
for tid in sorted(set(topics)):
    if shown >= 12:
        break
    subset = df[df["bertopic_id"] == tid]
    kw = topic_keywords.get(tid, "")
    print(f"--- TÓPICO {tid} ({len(subset)} tweets) ---")
    print(f"    Keywords: {kw}")
    for _, row in subset.head(2).iterrows():
        print(f"    • {str(row['text'])[:200]}")
    print()
    shown += 1


## Guardado intermedio (post-BERTopic)

Por si la fase de zero-shot falla, al menos BERTopic queda guardado.


In [ ]:
df.to_csv("../data/raw/scam_us_v8_with_bertopic.csv", index=False, encoding="utf-8")
print(f"✓ Guardado intermedio: scam_us_v8_with_bertopic.csv ({len(df)} filas)")


## FASE 2 — Clasificación Zero-Shot con BART-MNLI

⏳ Esta es la parte lenta (30-90 min en CPU). Las 14 categorías de tu memoria del TFM + "no relacionado":


In [ ]:
CANDIDATE_LABELS = [
    "investment scam or cryptocurrency fraud",
    "romance scam",
    "phishing or identity theft",
    "government impersonation scam (IRS, USPS)",
    "bank fraud or wire transfer fraud",
    "payment app scam (Zelle, Venmo, Cash App)",
    "Ponzi or pyramid scheme",
    "tech support scam",
    "employment or job scam",
    "charity or donation scam",
    "insurance fraud",
    "corporate or securities fraud",
    "tax fraud or tax evasion",
    "not related to financial fraud",
]

LABEL_TO_CODE = {
    "investment scam or cryptocurrency fraud": "investment_crypto",
    "romance scam": "romance",
    "phishing or identity theft": "phishing_identity",
    "government impersonation scam (IRS, USPS)": "gov_impersonation",
    "bank fraud or wire transfer fraud": "bank_wire",
    "payment app scam (Zelle, Venmo, Cash App)": "payment_app",
    "Ponzi or pyramid scheme": "ponzi_pyramid",
    "tech support scam": "tech_support",
    "employment or job scam": "employment",
    "charity or donation scam": "charity",
    "insurance fraud": "insurance",
    "corporate or securities fraud": "corporate",
    "tax fraud or tax evasion": "tax",
    "not related to financial fraud": "not_related",
}

print(f"Categorías candidatas: {len(CANDIDATE_LABELS)}")
for i, lbl in enumerate(CANDIDATE_LABELS, 1):
    print(f"  {i:2d}. {lbl}")


In [ ]:
print("⏳ Cargando modelo BART-MNLI (~1.6 GB, primera vez tarda 5-10 min)...")
t0 = time.time()

from transformers import pipeline
import torch

# En MacBook Air Intel → CPU
device = -1
print(f"   Dispositivo: CPU")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
)

t_load = time.time() - t0
print(f"✓ Modelo cargado en {t_load/60:.1f} min")


In [ ]:
# Clasificación con guardado incremental cada 100 tweets
texts_to_classify = df["clean_text"].tolist()
n = len(texts_to_classify)
BATCH_SIZE = 8
CHECKPOINT_EVERY = 100

predictions = []
scores = []

print(f"⏳ Clasificando {n} tweets en {len(CANDIDATE_LABELS)} categorías...")
print(f"   Lote: {BATCH_SIZE} | Checkpoint cada {CHECKPOINT_EVERY}")
print(f"   ESTIMACIÓN: 30-90 min en CPU. Sé paciente.\n")

t0 = time.time()
last_print = t0

for i in range(0, n, BATCH_SIZE):
    batch = texts_to_classify[i:i+BATCH_SIZE]

    try:
        results = classifier(
            batch,
            candidate_labels=CANDIDATE_LABELS,
            multi_label=False,
        )
        if isinstance(results, dict):
            results = [results]
    except Exception as e:
        print(f"  ⚠️ Error en lote {i}: {e}")
        for _ in batch:
            predictions.append("not related to financial fraud")
            scores.append(0.0)
        continue

    for r in results:
        predictions.append(r["labels"][0])
        scores.append(r["scores"][0])

    if (time.time() - last_print) > 30 or (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0:
        elapsed = time.time() - t0
        done = min(i + BATCH_SIZE, n)
        eta = (elapsed / done) * (n - done) if done > 0 else 0
        print(f"  {done}/{n} ({done/n*100:.1f}%) — {elapsed/60:.1f} min — ETA: {eta/60:.1f} min")
        last_print = time.time()

    # Checkpoint cada 100
    if (i + BATCH_SIZE) % CHECKPOINT_EVERY == 0 and (i + BATCH_SIZE) < n:
        df_partial = df.iloc[:len(predictions)].copy()
        df_partial["predicted_label"] = predictions
        df_partial["confidence_score"] = scores
        df_partial.to_csv("../data/raw/scam_us_v8_classification_checkpoint.csv", index=False)

t_classify = time.time() - t0
print(f"\n✓ Clasificación completada en {t_classify/60:.1f} min")


In [ ]:
# Añadir predicciones al DataFrame
df["predicted_label"] = predictions[:len(df)]
df["confidence_score"] = scores[:len(df)]
df["predicted_category"] = df["predicted_label"].map(LABEL_TO_CODE)
df["is_relevant"] = df["predicted_category"] != "not_related"

# Distribución
print("=== DISTRIBUCIÓN DE CATEGORÍAS PREDICHAS ===\n")
counts = df["predicted_category"].value_counts()
total = len(df)
for cat, n_cat in counts.items():
    pct = n_cat / total * 100
    print(f"  {cat:<22} {n_cat:>5}  ({pct:>5.1f}%)")

print(f"\nRelevantes (no 'not_related'): {df['is_relevant'].sum()} / {total} ({df['is_relevant'].mean()*100:.1f}%)")


In [ ]:
# Distribución de confianza
print("=== CONFIANZA EN PREDICCIONES ===\n")
print(f"  Media:    {df['confidence_score'].mean():.3f}")
print(f"  Mediana:  {df['confidence_score'].median():.3f}")
print(f"  Min:      {df['confidence_score'].min():.3f}")
print(f"  Max:      {df['confidence_score'].max():.3f}")
print(f"\n  Alta confianza (≥0.7):  {(df['confidence_score'] >= 0.7).sum()}")
print(f"  Media (0.4-0.7):        {((df['confidence_score'] >= 0.4) & (df['confidence_score'] < 0.7)).sum()}")
print(f"  Baja (<0.4):            {(df['confidence_score'] < 0.4).sum()}")


## Guardado final

In [ ]:
OUTPUT = "../data/raw/scam_us_FINAL_classified.csv"
df.to_csv(OUTPUT, index=False, encoding="utf-8")

print(f"✓ Guardado: {OUTPUT}")
print(f"  Total filas: {len(df)}")
print()
print("📦 Archivos generados:")
print(f"   scam_us_v8_with_bertopic.csv         ({len(df)} filas - solo BERTopic)")
print(f"   scam_us_v8_classification_checkpoint.csv (checkpoints intermedios)")
print(f"   scam_us_FINAL_classified.csv         ({len(df)} filas - clasificado completo)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del scam_us_FINAL_classified.csv en Drive.")
print()
print("📌 LISTO PARA LA FASE DE ANÁLISIS DE TU TFM.")


## Inspección de resultados

In [ ]:
print("=== 3 EJEMPLOS DE CADA CATEGORÍA (alta confianza) ===\n")
for cat_code in df["predicted_category"].value_counts().index[:10]:
    subset = df[
        (df["predicted_category"] == cat_code) &
        (df["confidence_score"] >= 0.5)
    ].head(3)
    n_total = (df["predicted_category"] == cat_code).sum()
    print(f"--- {cat_code} ({n_total} total) ---")
    for _, row in subset.iterrows():
        print(f"  [{row['confidence_score']:.2f}] {str(row['text'])[:200]}")
    print()


In [ ]:
# Tweets clasificados como "not_related" - verifica que están bien descartados
nr_count = (df["predicted_category"] == "not_related").sum()
if nr_count > 0:
    print(f"=== EJEMPLOS DE 'not_related' ({nr_count} en total) ===\n")
    not_related = df[df["predicted_category"] == "not_related"].sample(
        min(10, nr_count), random_state=42,
    )
    for _, row in not_related.iterrows():
        print(f"  [{row['confidence_score']:.2f}] @{row['username']}")
        print(f"    {str(row['text'])[:200]}")
        print()
else:
    print("No hay tweets en 'not_related'.")
